# 02 - Feature engineering

## Revision history (both revisions were caused by verification, and both matter)

**v1 -> v2: a leak found and killed.** v1 built 28-day chronic load and the
acute:chronic workload ratio from a reconstructed daily series. Those features drove
validation AUC to 0.987 - impossible on a 1.4%-prevalence problem, and our own rule
says treat >0.9 as presumptive leakage. Autopsy: `km_28d`, `acwr`, `rest_days_14d`
and `wow_km_change` were NaN for **100% of injury rows** vs 4-19% of healthy rows,
and one missing-indicator flag carried 93% of XGBoost's gain. Root cause was the
dataset's sampling scheme (section 2 below), not our code. Chronic load and ACWR are
unbuildable here without artifact, in any framing.

**v2 -> v3: the source paper corrected our window alignment.** Lövdal et al. (2021)
state the feature window covers the 7 days **before** the event - "the day before the
event is seen as day 0 ... 7 days before the event is day 6" - and the prediction
target is whether the *next* training session results in injury. Combined with 01's
proof that slot `.6` is the newest slot, that means **`.6` is the day before the
injury, not the injury day itself.** v2 had dropped `.6` from the headline features as
a leakage precaution; that precaution was based on a wrong assumption of ours, so v3
uses all 7 slots - which is also exactly the paper's setup, making the benchmark
comparison direct. The 6-slot variant survives as a labeled 2-day-lead sensitivity
check.

v3 also adopts the paper's **per-athlete normalization**, in a deliberately
leak-free form (section 4).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RAW = Path('../data/raw')
OUT = Path('../data/processed')
OUT.mkdir(exist_ok=True)

BASE = ['nr. sessions', 'total km', 'km Z3-4', 'km Z5-T1-T2', 'km sprinting',
        'strength training', 'hours alternative', 'perceived exertion',
        'perceived trainingSuccess', 'perceived recovery']

day = pd.read_csv(RAW / 'day_approach_maskedID_timeseries.csv.gz')
print(day.shape, '| positives:', int(day.injury.sum()))

(42766, 73) | positives: 583


## 1. Daily-series reconstruction (kept: it is how the artifact was found)

Slot k covers day t-(6-k), so `.6` is the newest of the 7 pre-event days and the
unsuffixed block the oldest. Windows overlap, so the daily series is recoverable and
every overlapping copy must agree - a structural check on the whole reading.

In [2]:
blocks = []
for k in range(7):
    cols = BASE if k == 0 else [f'{b}.{k}' for b in BASE]
    blk = day[cols].copy()
    blk.columns = BASE
    blk['Athlete ID'] = day['Athlete ID'].values
    blk['day'] = day['Date'].values - (6 - k)
    blocks.append(blk)
long = pd.concat(blocks, ignore_index=True)
g = long.groupby(['Athlete ID', 'day'])[BASE]
spread = (g.max() - g.min()).abs().to_numpy().max()
print('slot records:', len(long), '| max disagreement across overlaps:', spread)
assert spread < 1e-9
g.first().reset_index().to_csv(OUT / 'daily_series.csv.gz', index=False)
print('daily records written:', len(g.first()))

slot records: 299362 | max disagreement across overlaps: 0.0


daily records written: 49162


## 2. The publisher's healthy-event buffer - why long windows stay dead

The paper's Methods: *"For the healthy events, we demanded that the athlete is fully
fit 3 weeks before and 3 weeks after the event day"*, and injuries within 3 weeks of
a previous injury were filtered out as the same injury. Our measurement of the raw
rows found exactly that footprint before we had read it:

In [3]:
d = day.sort_values(['Athlete ID', 'Date'])
d['prev_row_gap'] = d.groupby('Athlete ID').Date.diff()
pos, neg = d[d.injury == 1], d[d.injury == 0]
print('days since the same athlete\'s previous row:')
print('  injury rows : min %d, median %.0f | with a row <=7d before: %d of %d'
      % (pos.prev_row_gap.min(), pos.prev_row_gap.median(),
         int((pos.prev_row_gap <= 7).sum()), len(pos)))
print('  healthy rows: median %.0f | with a row <=7d before: %.1f%%'
      % (neg.prev_row_gap.median(), 100 * (neg.prev_row_gap <= 7).mean()))

days since the same athlete's previous row:
  injury rows : min 22, median 22 | with a row <=7d before: 0 of 583
  healthy rows: median 1 | with a row <=7d before: 98.9%


**Consequence.** Any feature needing data from before the row's own 7-day window is
systematically unavailable for injury rows and available for healthy rows, so every
possible handling of that hole - flags, imputation, zero-fill, or dropping incomplete
rows (which would delete all 583 positives) - encodes the label. Chronic load, ACWR,
14-day rest patterns and week-over-week change are therefore out of scope on this
dataset. That is a real modeling cost, stated rather than hidden.

## 3. Raw window features (two window lengths)

Slot map: `.6` = day before the event, ..., unsuffixed = 7 days before.

- **7-slot (headline, `__w7`)**: all 7 pre-event days - the paper's exact window
- **6-slot (sensitivity, `__w6`)**: drops `.6`, giving 2 days of lead time instead
  of 1 - a "would this still work with more warning?" check, not a leakage fix

12 features per window: `km_sum`, `sessions`, `km_mod`, `km_hi`, `pct_mod`, `pct_hi`,
`rest_days`, `strength_n`, `alt_hours`, `exertion_avg`, `recovery_avg`,
`success_avg`. The two `pct_*` features are NaN when the window has zero running km -
a value-based, symmetric kind of missing (handled by impute+flag in 03).

In [4]:
def window_features(slots, suffix):
    pick = lambda b, k: day[b] if k == 0 else day[f'{b}.{k}']
    stack = {b: pd.concat([pick(b, k) for k in slots], axis=1) for b in BASE}
    f = pd.DataFrame(index=day.index)
    f['km_sum']     = stack['total km'].sum(axis=1)
    f['sessions']   = stack['nr. sessions'].sum(axis=1)
    f['km_mod']     = stack['km Z3-4'].sum(axis=1)
    f['km_hi']      = stack['km Z5-T1-T2'].sum(axis=1) + stack['km sprinting'].sum(axis=1)
    f['pct_mod']    = f['km_mod'] / f['km_sum'].replace(0, np.nan)
    f['pct_hi']     = f['km_hi']  / f['km_sum'].replace(0, np.nan)
    f['rest_days']  = (stack['nr. sessions'] == 0).sum(axis=1)
    f['strength_n'] = stack['strength training'].sum(axis=1)
    f['alt_hours']  = stack['hours alternative'].sum(axis=1)
    f['exertion_avg'] = stack['perceived exertion'].mean(axis=1)
    f['recovery_avg'] = stack['perceived recovery'].mean(axis=1)
    f['success_avg']  = stack['perceived trainingSuccess'].mean(axis=1)
    return f.add_suffix(suffix)

w7 = window_features(range(0, 7), '__w7')   # all 7 pre-event days (paper)
w6 = window_features(range(0, 6), '__w6')   # drops the day before the event
X = pd.concat([day[['Athlete ID', 'Date', 'injury']], w7, w6], axis=1)
print(X.shape)

(42766, 27)


## 4. Per-athlete normalization - adopted from the paper, made leak-free

The paper z-normalizes each feature per athlete using that athlete's healthy events.
This is probably their single biggest performance driver, because it converts "90 km
this week" into "heavy *for this runner*" - and with athletes ranging 43 to 1,791
logged rows and wildly different volumes, absolute km means different things to
different people.

**Our deviation, deliberate and documented.** Their per-athlete statistics are
computed over all of an athlete's healthy events, including ones *after* the row being
normalized. That is mildly transductive - a row's features depend on future data - so
copying it exactly would undercut every leakage guarantee this repo has built. Instead
each row is normalized by an **expanding window over that athlete's strictly earlier
healthy rows** (shifted by one, so the row itself never contributes):

`z = (x - mean(past healthy x)) / sd(past healthy x)`

Requires >= 20 prior healthy rows, else NaN. Expect a real cost - early rows in each
athlete's history cannot be normalized - and the NaN audit below has to show that
cost falls roughly evenly on injury and healthy rows, or this whole feature family is
as poisonous as v1's chronic load was.

In [5]:
MIN_HIST = 20
RAWF = [c for c in X.columns if c.endswith('__w7') or c.endswith('__w6')]
X = X.sort_values(['Athlete ID', 'Date']).reset_index(drop=True)

znorm = {}
for aid, a in X.groupby('Athlete ID', sort=False):
    healthy = a.injury == 0
    for col in RAWF:
        vals = a[col]
        h = vals.where(healthy)                     # healthy rows only
        past_mean = h.shift(1).expanding().mean()   # strictly earlier rows
        past_sd   = h.shift(1).expanding().std()
        past_n    = h.shift(1).expanding().count()
        z = (vals - past_mean) / past_sd.replace(0, np.nan)
        znorm.setdefault(col, []).append(z.where(past_n >= MIN_HIST))

Z = pd.DataFrame({f'{col}__z': pd.concat(parts).sort_index()
                  for col, parts in znorm.items()})
X = pd.concat([X, Z], axis=1)
print('normalized features added:', Z.shape[1])

normalized features added: 24


In [6]:
# NaN SYMMETRY AUDIT - the check that would have caught v1's artifact at birth.
# Any feature family whose missingness tracks the label is disqualified.
featcols = [c for c in X.columns if '__' in c]
audit = pd.DataFrame({'pos_nan': X[X.injury == 1][featcols].isna().mean(),
                      'neg_nan': X[X.injury == 0][featcols].isna().mean()})
audit['gap'] = (audit.pos_nan - audit.neg_nan).abs()
print(audit[audit.max(axis=1) > 0].sort_values('gap', ascending=False).round(4).to_string())
worst = audit.gap.max()
print(f'\nworst |pos_nan - neg_nan| across all features: {worst:.4f}')
print('v1 comparison: that gap was 0.81-0.96 on the chronic-load features')

                     pos_nan  neg_nan     gap
pct_mod__w6           0.0343   0.1088  0.0745
pct_hi__w6            0.0343   0.1088  0.0745
pct_hi__w6__z         0.1286   0.1982  0.0696
pct_mod__w7           0.0343   0.1019  0.0676
pct_hi__w7            0.0343   0.1019  0.0676
pct_hi__w7__z         0.1286   0.1920  0.0634
pct_mod__w6__z        0.1098   0.1675  0.0578
pct_mod__w7__z        0.1098   0.1608  0.0510
strength_n__w6__z     0.1029   0.1438  0.0409
strength_n__w7__z     0.1012   0.1411  0.0399
km_hi__w6__z          0.0943   0.1253  0.0310
km_hi__w7__z          0.0943   0.1249  0.0305
exertion_avg__w7__z   0.0549   0.0398  0.0151
recovery_avg__w7__z   0.0549   0.0398  0.0151
sessions__w7__z       0.0549   0.0398  0.0151
sessions__w6__z       0.0549   0.0399  0.0150
exertion_avg__w6__z   0.0549   0.0399  0.0150
recovery_avg__w6__z   0.0549   0.0399  0.0150
success_avg__w7__z    0.0549   0.0400  0.0149
success_avg__w6__z    0.0549   0.0401  0.0148
rest_days__w7__z      0.0566   0.0

In [7]:
# Does normalization actually separate the classes better than raw? A quick
# univariate look (AUC of each single feature), not a result - 03/04 decide.
from sklearn.metrics import roc_auc_score
rows = []
for base in ['km_sum', 'km_hi', 'exertion_avg', 'recovery_avg', 'success_avg', 'rest_days']:
    for variant, col in [('raw w7', f'{base}__w7'), ('z-norm w7', f'{base}__w7__z')]:
        s = X[[col, 'injury']].dropna()
        rows.append({'feature': base, 'variant': variant,
                     'n': len(s), 'univariate AUC': roc_auc_score(s.injury, s[col])})
print(pd.DataFrame(rows).pivot(index='feature', columns='variant',
                               values='univariate AUC').round(3).to_string())

variant       raw w7  z-norm w7
feature                        
exertion_avg   0.628      0.625
km_hi          0.582      0.579
km_sum         0.537      0.598
recovery_avg   0.603      0.605
rest_days      0.434      0.403
success_avg    0.598      0.633


In [8]:
X.to_csv(OUT / 'features_day.csv.gz', index=False)
print('wrote', OUT / 'features_day.csv.gz', X.shape)
print('feature families: __w7 (paper window), __w6 (2-day lead), __z (per-athlete)')

wrote ../data/processed/features_day.csv.gz (42766, 51)
feature families: __w7 (paper window), __w6 (2-day lead), __z (per-athlete)


## Summary

- Window alignment corrected from the source paper: all 7 slots are pre-event days,
  so the headline window is 7 slots (paper-identical) and the 6-slot variant is a
  lead-time sensitivity check, not a leakage fix.
- Publisher's 3-week healthy-event buffer confirmed in the paper's own Methods and
  measured in the raw rows; chronic load / ACWR remain out of scope with the reason
  documented.
- Per-athlete z-normalization adopted from the paper but computed over each
  athlete's strictly earlier healthy rows (>= 20 required), so no row is normalized
  using its own future. The deviation from the paper is deliberate and logged.
- NaN symmetry audit is now a standing gate on every feature family.